In [ ]:
from keras.applications import DenseNet169
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping
import cv2
import numpy as np
from keras.layers import GlobalAveragePooling2D, Dense, Lambda
from keras.models import Model
from keras.regularizers import l2
import keras.backend as K

In [ ]:
# Define image size for DenseNet121
img_rows, img_cols = 224, 224 

# Load pre-trained DenseNet121 model
DenseNet169 = DenseNet169(weights='imagenet', 
                          include_top=False, 
                          input_shape=(img_rows, img_cols, 3))

# Freeze all layers in DenseNet121
for layer in DenseNet169.layers:
    layer.trainable = False


In [ ]:
# Define the data directories
import os
train_data_dir = 'trainap'
validation_data_dir = 'testap'

# Remove .ipynb_checkpoints folder
train_folders = [folder for folder in os.listdir(train_data_dir) if folder != '.ipynb_checkpoints']
validation_folders = [folder for folder in os.listdir(validation_data_dir) if folder != '.ipynb_checkpoints']

# Create data generators for training and validation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

validation_datagen = ImageDataGenerator(rescale=1./255)

batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    classes=train_folders)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    classes=validation_folders)

In [ ]:
# Create top layers function
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Lambda,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K
# Function to create the top layers of the model
from keras.regularizers import l2
# Function to create the top layers of the model
def create_top_layers(bottom_model, num_classes,reg_strength=0.001):
    regularizer = l2(reg_strength)
    #top_model = bottom_model.output
    top_model = GlobalAveragePooling2D()(bottom_model.output, training=False)
    top_model = Dense(1664, activation='relu', kernel_regularizer=regularizer)(top_model)
    top_model = Dense(882, activation='relu', kernel_regularizer=regularizer)(top_model)
    #top_model = Dropout(0.5)(top_model)
    #top_model = Dense(512, activation='relu', kernel_regularizer=regularizer)(top_model)
    #top_model = Dropout(0.5)(top_model)
    top_model = Dense(440, activation='relu', kernel_regularizer=regularizer)(top_model)
    top_model = Dense(220, activation='relu', kernel_regularizer=regularizer)(top_model)
    top_model = Dropout(0.5)(top_model)
    #top_model = Dense(128, activation='relu')(top_model)

    # Adding L2 normalization
    #top_model = Lambda(lambda x: K.l2_normalize(x, axis=1))(top_model)

    top_model = Dense(num_classes, activation='softmax')(top_model)
    #top_model = ArcFaceLayer(num_classes)([top_model, labels])
    return top_model

num_classes = len(train_folders)

# Create the top layers using EfficientNetB0 as the bottom model
top_layers = create_top_layers(DenseNet169, num_classes)

# Create the final model
model = Model(inputs=DenseNet169.input, outputs=top_layers)

# Print model summary
model.summary()

In [ ]:
import matplotlib.pyplot as plt

# Load one batch of augmented training data
batch = train_generator.next()

# Display some of the augmented images
plt.figure(figsize=(10, 10))
for i in range(batch[0].shape[0]):
    plt.subplot(3, 3, i+1)
    plt.imshow(batch[0][i])
    plt.axis('off')
plt.show()


In [ ]:
# Define callbacks for model training
checkpoint = ModelCheckpoint("DenseNet169.h5",
                             monitor="val_loss",
                             mode="min",
                             save_best_only=True,
                             verbose=2)

#earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=5,verbose=1,restore_best_weights=True)
from keras.callbacks import ReduceLROnPlateau
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
callbacks=[checkpoint, reduce_lr]

In [ ]:
# Get the class names and class indices from the training generator
class_names = train_generator.class_indices

# Print the class names and their indices
for class_name, class_index in class_names.items():
    print(f"Class Name: {class_name}, Class Index: {class_index}")

In [ ]:

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])



history = model.fit(
    train_generator,
    epochs = 150,
    callbacks = callbacks,
    validation_data = validation_generator, verbose=2)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.title("DenseNet169 Accuracy History")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.title("Loss History")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K

model = load_model('DenseNet169.h5')
# Get predictions for the test set
y_pred = model.predict(validation_generator)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = validation_generator.classes

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot the confusion matrix using seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Inception-ResNetV2 Confusion Matrix')
plt.show()


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
import numpy as np
import os

# Load your trained VGG16 model
# Replace 'your_model.h5' with the actual filename of your saved model
your_model = load_model('DenseNet169.h5')
# Define the data directories
test_data_dir = 'test1'
#test_data_dir = 'check'
# Remove .ipynb_checkpoints folder

validation_folders = [folder for folder in os.listdir(test_data_dir) if folder != '.ipynb_checkpoints']

# Set image size based on your model's input size
img_size = (224, 224)

# Create data generators for testing
test_datagen = ImageDataGenerator(rescale=1./255)

batch_size = 32

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    classes=validation_folders,# or 'binary' depending on your problem
    shuffle=False  # set to False to get predictions in the order of the filenames
)

# Get predictions from the model
predictions = your_model.predict(test_generator)

# Convert predictions to class labels
predicted_classes = np.argmax(predictions, axis=1)

# Get true labels
true_labels = test_generator.classes

# Calculate accuracy
accuracy = np.sum(predicted_classes == true_labels) / len(true_labels)

print(f'Test Accuracy: {accuracy * 100:.2f}%')


In [ ]:
from keras.models import load_model
import cv2
import numpy as np

# Load the trained model
classifier = load_model('densenet121.h5')

# Define the dictionary for mapping class indices to class names
monkey_breeds_dict = {
    0: "pervez",
    1: "rupom",
    2: "aabir"
}

def draw_test(name, pred, im):
    monkey = monkey_breeds_dict[pred]
    BLACK = [0, 0, 0]
    expanded_image = cv2.copyMakeBorder(im, 80, 0, 0, 100, cv2.BORDER_CONSTANT, value=BLACK)
    cv2.putText(expanded_image, monkey, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow(name, expanded_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Provide the image path for prediction
img_path = "rupom.jpg"

# Read the image
input_im = cv2.imread(img_path)
input_original = input_im.copy()
input_original = cv2.resize(input_original, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)

input_im = cv2.resize(input_im, (224, 224), interpolation=cv2.INTER_LINEAR)
input_im = input_im / 255.
input_im = input_im.reshape(1, 224, 224, 3)

# Get Prediction
res = np.argmax(classifier.predict(input_im, 1, verbose=0), axis=1)

# Show image with predicted class
draw_test("Prediction", res[0], input_original)


In [ ]:
#Final version but the fps is not working perfectly



import cv2
import numpy as np
import threading
from keras.models import load_model
import time
# Load the trained model
classifier = load_model('densenet121.h5')

# Define the dictionary for mapping class indices to class names
monkey_breeds_dict = {
    0: "unkn",
    1: "pervez",
    2: "rupom",
    3: "aabir"
}

# Initialize video capture
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)

# Define a function for face detection and prediction
def detect_and_predict(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face = cv2.resize(face, (224, 224))
        face = face / 255.
        face = np.expand_dims(face, axis=0)
        
        pred = classifier.predict(face)
        pred_class = np.argmax(pred, axis=1)
        confidence = np.max(pred) * 100
        if confidence > 80:  # Set a threshold for recognizing
            monkey = monkey_breeds_dict[pred_class[0]]
            text = f"{monkey} ({confidence:.2f}%)"
            color = (0, 255, 0)
        else:
            text = "unknown"
            color = (0, 0, 255)
        
        
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return frame

# Initialize Haarcascades face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def video_thread():
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        processed_frame = detect_and_predict(frame)
        
        cv2.putText(processed_frame, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow('Video', processed_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Start the video thread
video_thread = threading.Thread(target=video_thread)
video_thread.start()

# Wait for the video thread to finish
video_thread.join()

# Release the video capture
cap.release()
cv2.destroyAllWindows()
